# 1-1 파도: 수식 없이 보기

**하는 법**
- 코드 셀은 읽지 않아도 됩니다. 위에서부터 `Shift+Enter`로 실행하고 **그림만** 보세요.
- 🤔 표시가 있으면 **실행하기 전에** A/B/C 중 하나를 골라 보세요. 감으로 골라도 됩니다. 틀리는 게 정상입니다.

In [ ]:
# 준비 (읽지 않아도 됨)
%load_ext autoreload
%autoreload 2
import time
from dataclasses import replace
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, animation
from IPython.display import HTML
_kfont = Path("/mnt/c/Windows/Fonts/malgun.ttf")
if _kfont.exists():
    font_manager.fontManager.addfont(str(_kfont)); plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["animation.embed_limit"] = 50

from rescue_sim import CONFIG_DIR
from rescue_sim.ocean import OceanConfig, WaveField
from rescue_sim.ocean.spectrum import sample_components

harbor = OceanConfig.from_yaml(CONFIG_DIR / "ocean.yaml", preset="harbor").waves
rough  = OceanConfig.from_yaml(CONFIG_DIR / "ocean.yaml", preset="rough").waves
X, Y = np.meshgrid(np.linspace(-25, 25, 128), np.linspace(-25, 25, 128))

def top_view(ax, cfg, title, comp=None, t=0.0, vmax=None):
    """바다를 위에서 내려다본 그림. 빨강 = 솟은 곳(마루), 파랑 = 꺼진 곳(골)."""
    eta = WaveField(cfg, components=comp, depth=5.0).elevation(X, Y, t)
    v = vmax or np.abs(eta).max()
    ax.imshow(eta, extent=[-25, 25, -25, 25], origin="lower", cmap="RdBu_r", vmin=-v, vmax=v)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
print("준비 완료")

---
## 1. 바다는 '단순한 물결'을 여러 개 겹친 것

조약돌 하나를 연못에 던지면 **규칙적인 물결**이 생깁니다. 빨래판처럼 똑같은 줄무늬가 반복되는 모양입니다.

진짜 바다는 이런 물결을 **크기·간격·방향이 제각각인 것들로 수십 개** 겹친 것입니다. 우리 코드도 정확히 이렇게 합니다. 기본값은 **32개**입니다.

🤔 **예측:** 물결을 **1개, 3개, 32개** 겹친 바다를 위에서 내려다보면?
- A. 셋 다 비슷한 줄무늬
- B. 1개는 줄무늬, 개수가 늘수록 불규칙한 '진짜 바다'처럼 보인다
- C. 개수가 늘수록 더 규칙적이 된다

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 4.3))
for a, n in zip(ax, [1, 3, 32]):
    top_view(a, replace(harbor, n_components=n), f"물결 {n}개 겹침")
plt.suptitle("위에서 본 50m × 50m 바다  (빨강 = 솟은 곳, 파랑 = 꺼진 곳)");

💡 겹치는 개수가 많을수록 진짜 바다처럼 보입니다. 대신 **계산이 그만큼 늘어납니다.** 이 문제는 맨 마지막 6번에서 다시 봅니다.

---
## 2. 바다 성격을 정하는 다이얼 3개

설정 파일 `config/ocean.yaml`에서 바다는 숫자 몇 개로 정해집니다.

| 다이얼 | 뜻 (일상 말로) | harbor | rough |
|---|---|---|---|
| **Hs** (파고) | 큰 파도들이 대략 이 정도 높이 | 0.2 m (무릎 아래) | 0.8 m (허리쯤) |
| **Tp** (주기) | 파도 하나가 지나가고 다음 파도가 오기까지 걸리는 시간 | 3초 | 6초 |
| **방향 퍼짐** | 파도가 한 방향에서만 오나, 여러 방향에서 섞여 오나 | 30° | 30° |

🤔 **예측:** **Tp(주기)만** 3초에서 6초로 늘리면, 위에서 본 바다는?
- A. 파도 사이 간격이 **넓어진다** (길고 느린 너울)
- B. 파도 사이 간격이 **좁아진다**
- C. 높이만 달라지고 모양은 같다

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
top_view(ax[0], replace(harbor, tp=3.0), "Tp = 3초", vmax=0.15)
top_view(ax[1], replace(harbor, tp=6.0), "Tp = 6초", vmax=0.15);

💡 주기가 길면 파도 사이 간격도 넓어집니다. 3초면 약 14m, 6초면 약 40m입니다. 그래서 rough 바다는 50m 영역 안에 큰 마루가 1~2개만 보입니다.

🤔 **예측:** **방향 퍼짐**을 30°에서 **1°**로 줄이면? (모든 물결이 거의 같은 방향에서 옴)
- A. 마루가 길고 곧은 **줄무늬**가 된다
- B. 점박이 무늬가 된다
- C. 변화 없다

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
top_view(ax[0], harbor, "방향 퍼짐 30°")
top_view(ax[1], replace(harbor, spreading_deg=1.0), "방향 퍼짐 1°");

---
## 3. 움직이는 바다

지금까지는 사진이었고, 이번엔 20초짜리 영상입니다. 파도는 **오른쪽(먼바다)에서 왼쪽(해안)으로** 오도록 설정돼 있습니다. 만드는 데 몇 초 걸립니다.

In [ ]:
wf = WaveField(harbor, depth=5.0)
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(wf.elevation(X, Y, 0), extent=[-25, 25, -25, 25], origin="lower", cmap="RdBu_r", vmin=-0.15, vmax=0.15)
ax.set_xlabel("← 해안          먼바다 →"); ax.set_yticks([])
def frame(i):
    im.set_data(wf.elevation(X, Y, i * 0.2)); ax.set_title(f"harbor  t = {i*0.2:4.1f}초"); return [im]
anim = animation.FuncAnimation(fig, frame, frames=100, interval=80)
plt.close(fig); HTML(anim.to_jshtml())

---
## 4. 부표 하나의 시점: MuJoCo가 그릴 수 있을까?

바다 한가운데 부표가 떠 있다고 생각하고, 60초 동안 부표가 위아래로 얼마나 움직이는지 그립니다.

MuJoCo의 수면은 **정지 수면 기준 위아래 0.5m까지만** 그릴 수 있게 만들어 두었습니다(`ocean_scene.xml`). 그래프의 빨간 점선이 이 한계입니다.

🤔 **예측:** 거친 바다(rough, Hs 0.8m)의 출렁임은 빨간 선 안에 들어올까?
- A. 둘 다 안에 들어온다
- B. harbor는 들어오고, rough는 빨간 선을 넘을 때가 있다
- C. 둘 다 넘는다

In [ ]:
t = np.arange(0, 60, 0.05)
plt.figure(figsize=(11, 3.5))
for name, cfg in [("harbor", harbor), ("rough", rough)]:
    wf = WaveField(cfg, depth=5.0)
    plt.plot(t, [wf.elevation(3.0, -2.0, ti) for ti in t], label=name)
for v in (0.5, -0.5): plt.axhline(v, ls="--", c="r", lw=1)
plt.xlabel("시간 [초]"); plt.ylabel("부표 높이 [m]"); plt.legend(); plt.title("부표의 오르내림 (빨간 점선 = MuJoCo 수면 한계)");

💡 빨간 선을 넘는 부분은 MuJoCo 화면에서 **잘려 납작하게** 보입니다. 1-2에서 고칠 첫 번째 문제입니다.

---
## 5. 망가뜨려 보기 (선택)

아래 셀에서 **숫자 하나만** 바꿔 실행해 보세요. 결과가 어떻게 바뀌는지 보는 게 목적입니다.

- `n_components=32` 를 `2`로
- `spreading_deg=30.0` 을 `90.0`으로
- `hs=0.2` 를 `0.8`로 (색은 자동으로 다시 맞춰지니 **오른쪽 색 막대 숫자**를 보세요)

In [ ]:
cfg = replace(harbor, n_components=32, spreading_deg=30.0, hs=0.2)   # ← 여기만 바꾸기
fig, ax = plt.subplots(figsize=(5, 5))
eta = WaveField(cfg).elevation(X, Y, 0.0)
im = ax.imshow(eta, extent=[-25, 25, -25, 25], origin="lower", cmap="RdBu_r")
fig.colorbar(im, label="높이 [m]"); ax.set_xticks([]); ax.set_yticks([]);

### 5-2. 출발 타이밍을 전부 맞추면?

32개 물결은 원래 **출발 타이밍이 제각각**입니다(랜덤). 만약 전부 같은 순간에 출발하면 어떻게 될까요?

🤔 **예측:**
- A. 별 차이 없다
- B. 한 지점에 마루가 한꺼번에 겹쳐 **비정상적으로 큰 파도**가 생긴다
- C. 바다가 평평해진다

In [ ]:
c = sample_components(harbor)
c.phase[:] = 0.0          # 출발 타이밍을 전부 똑같이
fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
top_view(ax[0], harbor, "타이밍 랜덤 (정상)", vmax=0.3)
top_view(ax[1], harbor, "타이밍 전부 같음", comp=c, vmax=0.3);

---
## 6. MuJoCo 관점: 이 계산은 얼마나 비싼가  ⭐ 1-2에서 제일 중요

MuJoCo 수면은 **바둑판 격자**입니다. 칸마다 "여기 물 높이는 얼마"를 계산해서 넣어 줘야 화면에 파도가 보입니다. 지금 설정은 **128 × 128칸**이고 **1초에 25번** 다시 그립니다.

참고로 파도 없이 MuJoCo 물리만 돌리면 **실제 시간보다 약 2000배 빠릅니다.** 현실의 1초가 지나는 동안 시뮬레이션 속 시간은 2000초가 흘러간다는 뜻입니다.

🤔 **예측:** 128 × 128 수면 계산을 붙이면 몇 배속까지 떨어질까?
- A. 거의 그대로 (약 2000배)
- B. 약 200배
- C. 약 3배쯤까지 확 떨어진다

(1배 아래로 떨어지면 **실제 시간보다 느려서** 화면이 버벅입니다.)

In [ ]:
wf = WaveField(harbor)
print("격자 크기     한 장 그리는 데     → 파도 계산만으로 낼 수 있는 최대 배속")
for res in [32, 64, 128, 256]:
    Xr, Yr = np.meshgrid(np.linspace(-25, 25, res), np.linspace(-25, 25, res))
    t0 = time.perf_counter()
    for i in range(20): wf.elevation(Xr, Yr, i * 0.04)
    ms = (time.perf_counter() - t0) / 20 * 1e3
    print(f"{res:>3} × {res:<3}     {ms:6.2f} ms         →  {1 / (ms * 25 / 1e3):7.1f} 배")

💡 이게 이 프로젝트의 목적인 **'MuJoCo의 한계 시험'**이 처음 드러나는 지점입니다. 1-2에서는 이 숫자를 보고 다음 중 무엇을 할지 정합니다.
- 격자를 줄일까?
- 1초에 25번보다 덜 그릴까?
- 화면에 그리는 수면과 물리 계산(부력)에 쓰는 수면을 따로 둘까?

---
## 정리: 본인 말로 한 줄씩

1. 바다는 ...
2. Hs는 ____, Tp는 ____
3. MuJoCo에 붙일 때 문제 두 가지: ____ , ____
